In [ ]:
import json
import pandas as pd

with open("finale_day1.json", "r", encoding="utf-8") as f:
    data = json.load(f)

df_kaggle = pd.DataFrame(data["discussions"])
df_kaggle["source"] = "kaggle"

df_kaggle.to_csv("kaggle_1.csv", index=False)

In [ ]:
import json
import pandas as pd

with open("finale_day3.json", "r", encoding="utf-8") as f:
    data = json.load(f)

df_kaggle = pd.DataFrame(data["discussions"])
df_kaggle["source"] = "kaggle"

df_kaggle.to_csv("kaggle_4.csv", index=False)

In [ ]:
import json
import pandas as pd

with open("finale_week1.json", "r", encoding="utf-8") as f:
    data = json.load(f)

df_kaggle = pd.DataFrame(data["discussions"])
df_kaggle["source"] = "kaggle"

df_kaggle.to_csv("kaggle_3.csv", index=False)

In [ ]:
import pandas as pd

df1 = pd.read_csv("kaggle_1.csv")
df2 = pd.read_csv("kaggle_3.csv")
df3 = pd.read_csv("kaggle_4.csv")

df_all = pd.concat([df1, df2, df3], ignore_index=True)
df_all.to_csv("kaggle_all_discussions.csv", index=False)

In [ ]:
import re
import pandas as pd

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)
    return text

# Load browse_df from 'browse_st.csv' as it was not defined.
browse_df = pd.read_csv("browse_st.csv")
df_kaggle = pd.read_csv("kaggle_all_discussions.csv")

# Apply cleaning to the 'Title with Domain' column and store in a new 'text' column for browse_df
browse_df["text"] = browse_df["Title with Domain"].apply(clean_text)

# Corrected 'kaggle_df' to 'df_kaggle' and applied cleaning to the 'title' column, storing in 'text'
df_kaggle["text"] = df_kaggle["title"].apply(clean_text)

In [ ]:
final_columns = [
    "text",        # fan comment / theory / review
    "sentiment",   # Positive / Neutral / Negative
    "sentiment_score",
    "theme",       # theory cluster (later)
    "episode",     # if available
    "source"       # "browseai" or "kaggle"
]

In [ ]:
browse_df = browse_df.rename(columns={
    "comment": "text"   # adjust if needed
})

browse_df["source"] = "browseai"

for col in final_columns:
    if col not in browse_df.columns:
        browse_df[col] = None

browse_df = browse_df[final_columns]

In [17]:
df_kaggle = df_kaggle.rename(columns={
    "review_text": "text"   # adjust to your file
})

df_kaggle["source"] = "kaggle"

for col in final_columns:
    if col not in df_kaggle.columns:
        df_kaggle[col] = None

df_kaggle = df_kaggle[final_columns]

In [18]:
final_df = pd.concat([browse_df, df_kaggle], ignore_index=True)
final_df.to_csv("stranger_things_final.csv", index=False)

In [19]:


print('Combined DataFrame created and saved to combined_discussions.csv')
print(f'Shape of combined_df: {final_df.shape}')
print('Columns in combined_df:')
print(final_df.columns)

Combined DataFrame created and saved to combined_discussions.csv
Shape of combined_df: (719, 18)
Columns in combined_df:
Index(['Position', 'Rank', 'Score', 'Title with Domain', 'Title', 'Domain',
       'Submission Info', 'Time Ago', 'Author', 'Comments Count', 'Post URL',
       'Author URL', 'text', 'sentiment', 'sentiment_score', 'theme',
       'episode', 'source'],
      dtype='object')


In [20]:
import pandas as pd

df = final_df.copy()

# Keep only what we need for NLP
model_df = df[["text", "sentiment", "sentiment_score", "source"]].dropna(subset=["text"])

In [21]:
model_df.shape
model_df.head()

,text,sentiment,sentiment_score,source
0,stranger things: tales from ‘85 | official tea...,NaN,NaN,NaN
1,discussionstranger things season 5 episode dis...,NaN,NaN,NaN
2,discussionthe cheering in this scene makes abs...,NaN,NaN,NaN
3,guys i found the door 🚪 in gym (i.redd.it),NaN,NaN,NaN
4,was barb right or wrong here? (i.redd.it),NaN,NaN,NaN


In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=1000,
    stop_words="english",
    ngram_range=(1, 2)
)

X = tfidf.fit_transform(model_df["text"])

In [23]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=5, random_state=42)
model_df["cluster"] = kmeans.fit_predict(X)

In [25]:
terms = tfidf.get_feature_names_out()

for i in range(5):
    print(f"\nCLUSTER {i}")
    cluster_terms = X[(model_df["cluster"] == i).values].sum(axis=0)
    top_terms = cluster_terms.A1.argsort()[-10:][::-1]
    print([terms[j] for j in top_terms])


CLUSTER 0
['self', 'strangerthings', 'self strangerthings', 'season', 'season self', 'vecna', 'did', 'finale self', 'finale', 'discussioni']

CLUSTER 1
['redd', 'fan', 'season', 'season redd', 'mike', 'scene', 'think', 'just', 'fanart', 'did']

CLUSTER 2
['love', '10', 'years', 'redd', 'felt', 's5', 'like', 'felt like', 'things redd', 'love season']

CLUSTER 3
['com', 'old reddit', 'reddit com', 'old', 'reddit', 'netflix', 'season', 's5', 'finale', 'fan']

CLUSTER 4
['stranger things', 'stranger', 'things', 'finale', 'things finale', 'season', 'things season', 'episode', 'strangerthings', 'self strangerthings']


In [26]:
cluster_theme_map = {
    0: "Eleven Survival Theories",
    1: "Ending Dissatisfaction",
    2: "Upside Down Lore",
    3: "Writing Criticism",
    4: "Nostalgia & Emotion"
}

model_df["theme"] = model_df["cluster"].map(cluster_theme_map)

In [27]:
model_df.to_csv("final_theory_dataset.csv", index=False)

In [28]:
model_df["theme"].value_counts()

,count
theme,
Eleven Survival Theories,201
Nostalgia & Emotion,177
Ending Dissatisfaction,170
Writing Criticism,156
Upside Down Lore,15


In [29]:
model_df.sample(5)[["text", "theme", "sentiment"]]

,text,theme,sentiment
125,fan artumh billy fanart and my oc... (i.redd.it),Ending Dissatisfaction,NaN
175,discussionhow did hopper get captured in seaso...,Eleven Survival Theories,NaN
707,self aware,Eleven Survival Theories,None
278,even in another universe… (i.redd.it),Ending Dissatisfaction,NaN
189,would the party have a chance against the smil...,Writing Criticism,NaN


In [30]:
from nltk.sentiment import SentimentIntensityAnalyzer
import nltk

nltk.download("vader_lexicon")

sia = SentimentIntensityAnalyzer()

def get_sentiment(text):
    score = sia.polarity_scores(str(text))["compound"]
    if score >= 0.05:
        return "Positive"
    elif score <= -0.05:
        return "Negative"
    else:
        return "Neutral"

model_df["sentiment"] = model_df["text"].apply(get_sentiment)

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


In [31]:
model_df["sentiment"].value_counts()

,count
sentiment,
Neutral,379
Positive,215
Negative,125


In [33]:
model_df.to_csv("final_theory_dataset.csv", index=False)

In [34]:
model_df.sample(5)[["text", "theme", "sentiment"]]

,text,theme,sentiment
390,"discussionsuch a shame out of these 3, andy is...",Ending Dissatisfaction,Negative
308,fan artmaking a stranger things inspired game ...,Nostalgia & Emotion,Positive
394,spoilersjust finished s5 (self.strangerthings),Eleven Survival Theories,Neutral
290,huge gaps in filming stranger things (self.str...,Nostalgia & Emotion,Positive
350,mike wheeler’s ending (self.strangerthings),Eleven Survival Theories,Neutral


# Task
Update the `model_df` by calculating and storing the VADER compound sentiment score in a new 'sentiment_score' column. Then, use this 'sentiment_score' to categorize and populate the 'sentiment' column with 'Positive', 'Negative', or 'Neutral' values. Finally, save the updated `model_df` to "final_theory_dataset.csv" and confirm that the file now contains the correct sentiment scores and themes.

## Calculate Sentiment Score

### Subtask:
Modify the sentiment analysis code to calculate and store the VADER compound sentiment score in a new 'sentiment_score' column within the 'model_df'.


**Reasoning**:
The subtask requires calculating and storing the VADER compound sentiment score in a new 'sentiment_score' column. I will define a function to extract the compound score from VADER's polarity scores and apply it to the 'text' column of the `model_df`.



In [35]:
def get_sentiment_score(text):
    return sia.polarity_scores(str(text))["compound"]

model_df["sentiment_score"] = model_df["text"].apply(get_sentiment_score)
print("Calculated VADER compound sentiment scores and stored them in 'sentiment_score' column.")

Calculated VADER compound sentiment scores and stored them in 'sentiment_score' column.


**Reasoning**:
Now that the 'sentiment_score' column is populated, I need to update the 'sentiment' column based on these scores, using the defined thresholds. This will align the categorical sentiment with the calculated numerical scores.



In [36]:
def categorize_sentiment(score):
    if score >= 0.05:
        return "Positive"
    elif score <= -0.05:
        return "Negative"
    else:
        return "Neutral"

model_df["sentiment"] = model_df["sentiment_score"].apply(categorize_sentiment)
print("Updated 'sentiment' column based on 'sentiment_score'.")

Updated 'sentiment' column based on 'sentiment_score'.


**Reasoning**:
The subtask explicitly requires saving the updated `model_df` to 'final_theory_dataset.csv'. This step will persist the changes made to the sentiment scores and categories.



In [37]:
model_df.to_csv("final_theory_dataset.csv", index=False)
print("Updated model_df saved to final_theory_dataset.csv.")

Updated model_df saved to final_theory_dataset.csv.


**Reasoning**:
To confirm that the 'final_theory_dataset.csv' file has been updated correctly with the new 'sentiment_score' and 'sentiment' columns, I will load the CSV back into a DataFrame and display a sample of the 'text', 'theme', 'sentiment', and 'sentiment_score' columns.



In [38]:
updated_model_df = pd.read_csv("final_theory_dataset.csv")
print("Confirmation of updated file content:")
print(updated_model_df.sample(5)[["text", "theme", "sentiment", "sentiment_score"]])

Confirmation of updated file content:
                                                  text  \
625  if they would just have heard the leaked gossi...   
37   discussionim not really excited for the cartoo...   
45   memewth are they looking at ??😭😭 (old.reddit.com)   
151  thought id make a few missing lego sticker acc...   
191  discussionthis is the most underrated original...   

                        theme sentiment  sentiment_score  
625       Nostalgia & Emotion  Negative          -0.4890  
37   Eleven Survival Theories  Negative          -0.3078  
45          Writing Criticism   Neutral           0.0000  
151    Ending Dissatisfaction  Negative          -0.2960  
191       Nostalgia & Emotion   Neutral           0.0000  


## Summary:

### Data Analysis Key Findings
*   VADER compound sentiment scores were successfully calculated and stored in a new 'sentiment\_score' column for each text entry in the `model_df`.
*   Sentiments were categorized into 'Positive', 'Negative', or 'Neutral' within the 'sentiment' column, based on the 'sentiment\_score':
    *   Scores greater than or equal to 0.05 were classified as 'Positive'.
    *   Scores less than or equal to -0.05 were classified as 'Negative'.
    *   Scores between -0.05 and 0.05 (exclusive) were classified as 'Neutral'.
*   The updated `model_df`, which now includes the 'sentiment\_score' and revised 'sentiment' columns, was successfully saved to "final\_theory\_dataset.csv".
*   Confirmation by reloading the file showed that the new sentiment scores and categories were correctly persisted in "final\_theory\_dataset.csv".

### Insights or Next Steps
*   The newly added 'sentiment\_score' and 'sentiment' columns enrich the dataset, allowing for quantitative and qualitative analysis of sentiment alongside existing thematic data.
*   The dataset is now ready for further analysis, such as exploring the distribution of sentiment across different themes, or identifying themes that are predominantly associated with positive or negative sentiment.
